In [1]:
# Fuchibol26 - Celda 1: Cargar los datos
# Importamos pandas, que es la librería para manejar tablas de datos

import pandas as pd
import os

# Definimos la ruta a nuestros archivos
# ".." significa "sube una carpeta" (salimos de notebooks/ hacia Fuchibol26/)
RUTA_RESULTADOS = os.path.join("..", "datos", "historicos", "resultados.csv")

# Cargamos el archivo CSV en una variable llamada df
# df es abreviatura de DataFrame (tabla de datos)
df = pd.read_csv(RUTA_RESULTADOS)

# Mostramos cuántos partidos tenemos
print(f"✅ Datos cargados correctamente")
print(f"📊 Total de partidos: {len(df):,}")
print(f"📋 Columnas: {list(df.columns)}")

✅ Datos cargados correctamente
📊 Total de partidos: 49,477
📋 Columnas: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


In [2]:
# Celda 2: Ver cómo se ven los datos
# .head() muestra las primeras 5 filas de la tabla

print("📋 Primeros 5 partidos de la historia:")
print(df.head())

print(f"\n📐 Tamaño de la tabla:")
print(f"   {df.shape[0]} filas (partidos)")
print(f"   {df.shape[1]} columnas (información por partido)")

📋 Primeros 5 partidos de la historia:
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  

📐 Tamaño de la tabla:
   49477 filas (partidos)
   9 columnas (información por partido)


In [3]:
# Celda 3: ¿Hay datos faltantes?
# Esto es importante antes de construir modelos
# Un dato faltante puede causar errores más adelante

print("🔍 Datos faltantes por columna:")
print(df.isnull().sum())

print(f"\n✅ Total de celdas vacías en todo el dataset: {df.isnull().sum().sum()}")

🔍 Datos faltantes por columna:
date           0
home_team      0
away_team      0
home_score    60
away_score    60
tournament     0
city           0
country        0
neutral        0
dtype: int64

✅ Total de celdas vacías en todo el dataset: 120


In [4]:
# Celda 4: Limpiar partidos sin resultado
# Los 60 partidos sin score son partidos futuros (aún no se juegan)
# Los eliminamos porque no podemos aprender de partidos sin resultado

print(f"Partidos ANTES de limpiar: {len(df):,}")

# Eliminamos filas donde home_score o away_score estén vacíos
df = df.dropna(subset=["home_score", "away_score"])

# Convertimos los goles a número entero (eran decimales: 2.0 → 2)
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)

# Convertimos la fecha al tipo correcto
df["date"] = pd.to_datetime(df["date"])

print(f"Partidos DESPUÉS de limpiar: {len(df):,}")
print(f"Partidos eliminados: 60 (eran partidos futuros sin resultado)")
print(f"\n✅ Datos limpios y listos")
print(f"\n📋 Cómo se ven ahora los goles (ya sin decimales):")
print(df[["date", "home_team", "away_team", "home_score", "away_score"]].head())

Partidos ANTES de limpiar: 49,477
Partidos DESPUÉS de limpiar: 49,417
Partidos eliminados: 60 (eran partidos futuros sin resultado)

✅ Datos limpios y listos

📋 Cómo se ven ahora los goles (ya sin decimales):
        date home_team away_team  home_score  away_score
0 1872-11-30  Scotland   England           0           0
1 1873-03-08   England  Scotland           4           2
2 1874-03-07  Scotland   England           2           1
3 1875-03-06   England  Scotland           2           2
4 1876-03-04  Scotland   England           3           0


In [5]:
# Celda 5: ¿En qué torneos se jugaron estos partidos?
# Necesitamos saber esto para filtrar solo los que nos interesan

print(f"🏆 Total de torneos distintos: {df['tournament'].nunique()}")
print(f"\n🔝 Top 20 torneos por cantidad de partidos:")
print(df["tournament"].value_counts().head(20))

🏆 Total de torneos distintos: 200

🔝 Top 20 torneos por cantidad de partidos:
tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                            976
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                          

In [6]:
# Celda 6: Quedarnos solo con partidos oficiales
# Los amistosos no reflejan el verdadero nivel de cada selección
# Para predecir el Mundial necesitamos partidos que "importen"

TORNEOS_OFICIALES = [
    "FIFA World Cup",
    "FIFA World Cup qualification",
    "UEFA Euro",
    "UEFA Euro qualification",
    "Copa América",
    "African Cup of Nations",
    "African Cup of Nations qualification",
    "AFC Asian Cup",
    "AFC Asian Cup qualification",
    "Gold Cup",
    "UEFA Nations League",
    "CONCACAF Nations League"
]

# Filtramos: solo nos quedamos con torneos de la lista
df_oficial = df[df["tournament"].isin(TORNEOS_OFICIALES)].copy()
df_oficial = df_oficial.reset_index(drop=True)

# Mostramos el resultado
print(f"Partidos totales:                {len(df):,}")
print(f"Partidos oficiales:              {len(df_oficial):,}")
print(f"Partidos descartados (amist.):   {len(df) - len(df_oficial):,}")

print(f"\n📊 Partidos por torneo oficial:")
print(df_oficial["tournament"].value_counts())

Partidos totales:                49,417
Partidos oficiales:              19,750
Partidos descartados (amist.):   29,667

📊 Partidos por torneo oficial:
tournament
FIFA World Cup qualification            8771
UEFA Euro qualification                 2824
African Cup of Nations qualification    2327
FIFA World Cup                           976
Copa América                             869
African Cup of Nations                   845
AFC Asian Cup qualification              829
UEFA Nations League                      658
CONCACAF Nations League                  422
AFC Asian Cup                            421
Gold Cup                                 420
UEFA Euro                                388
Name: count, dtype: int64


In [7]:
# Celda 7: Calcular estadísticas para cada selección
# Recorremos todos los partidos y contamos victorias, empates, derrotas y goles

def calcular_estadisticas(dataframe, seleccion):
    """
    Esta función recibe el nombre de una selección
    y devuelve sus estadísticas históricas.
    """
    # Partidos donde jugó de local
    local = dataframe[dataframe["home_team"] == seleccion]
    # Partidos donde jugó de visitante
    visitante = dataframe[dataframe["away_team"] == seleccion]

    total = len(local) + len(visitante)
    if total == 0:
        return None

    # Victorias (cuando anotó más goles que el rival)
    victorias = (
        (local["home_score"] > local["away_score"]).sum() +
        (visitante["away_score"] > visitante["home_score"]).sum()
    )

    # Empates
    empates = (
        (local["home_score"] == local["away_score"]).sum() +
        (visitante["home_score"] == visitante["away_score"]).sum()
    )

    # Derrotas = lo que sobra
    derrotas = total - victorias - empates

    # Goles a favor y en contra
    gf = int(local["home_score"].sum() + visitante["away_score"].sum())
    gc = int(local["away_score"].sum() + visitante["home_score"].sum())

    return {
        "seleccion":        seleccion,
        "partidos":         total,
        "victorias":        int(victorias),
        "empates":          int(empates),
        "derrotas":         int(derrotas),
        "goles_favor":      gf,
        "goles_contra":     gc,
        "diferencia_goles": gf - gc,
        "pct_victorias":    round((victorias / total) * 100, 1),
        "prom_goles":       round(gf / total, 2)
    }

# Obtener lista de todas las selecciones únicas
todas = pd.concat([
    df_oficial["home_team"],
    df_oficial["away_team"]
]).unique()

print(f"Calculando estadísticas para {len(todas)} selecciones...")

# Calcular para cada una y guardar en una lista
lista = []
for seleccion in todas:
    stats = calcular_estadisticas(df_oficial, seleccion)
    if stats:
        lista.append(stats)

# Convertir la lista en una tabla y ordenar por % de victorias
df_stats = pd.DataFrame(lista)
df_stats = df_stats.sort_values("pct_victorias", ascending=False)
df_stats = df_stats.reset_index(drop=True)
df_stats.index += 1  # Ranking desde 1

print(f"✅ Listo!")
print(f"\n🏆 TOP 15 selecciones por % de victorias:")
print(df_stats.head(15).to_string())

Calculando estadísticas para 223 selecciones...
✅ Listo!

🏆 TOP 15 selecciones por % de victorias:
        seleccion  partidos  victorias  empates  derrotas  goles_favor  goles_contra  diferencia_goles  pct_victorias  prom_goles
1         Germany       413        271       85        57          987           376               611           65.6        2.39
2           Spain       414        263       86        65          905           334               571           63.5        2.19
3            Iran       300        186       71        43          642           208               434           62.0        2.14
4     Netherlands       398        245       77        76          882           341               541           61.6        2.22
5          Brazil       469        283      100        86          998           416               582           60.3        2.13
6       Argentina       467        277      104        86          928           429               499           59.3    

In [8]:
# Celda 8: Ver estadísticas de cualquier selección
# Cambia el nombre para ver otras selecciones

SELECCION = "Mexico"

stats = calcular_estadisticas(df_oficial, SELECCION)

if stats:
    print(f"{'='*40}")
    print(f"  📊 {stats['seleccion'].upper()}")
    print(f"{'='*40}")
    print(f"  Partidos jugados:    {stats['partidos']}")
    print(f"  Victorias:           {stats['victorias']}")
    print(f"  Empates:             {stats['empates']}")
    print(f"  Derrotas:            {stats['derrotas']}")
    print(f"  Goles a favor:       {stats['goles_favor']}")
    print(f"  Goles en contra:     {stats['goles_contra']}")
    print(f"  Diferencia de goles: {stats['diferencia_goles']:+}")
    print(f"  % de victorias:      {stats['pct_victorias']}%")
    print(f"  Promedio goles:      {stats['prom_goles']}")

  📊 MEXICO
  Partidos jugados:    383
  Victorias:           222
  Empates:             77
  Derrotas:            84
  Goles a favor:       761
  Goles en contra:     347
  Diferencia de goles: +414
  % de victorias:      58.0%
  Promedio goles:      1.99


In [9]:
# Celda 8: Ver estadísticas de cualquier selección
# Cambia el nombre para ver otras selecciones

SELECCION = "Guatemala"

stats = calcular_estadisticas(df_oficial, SELECCION)

if stats:
    print(f"{'='*40}")
    print(f"  📊 {stats['seleccion'].upper()}")
    print(f"{'='*40}")
    print(f"  Partidos jugados:    {stats['partidos']}")
    print(f"  Victorias:           {stats['victorias']}")
    print(f"  Empates:             {stats['empates']}")
    print(f"  Derrotas:            {stats['derrotas']}")
    print(f"  Goles a favor:       {stats['goles_favor']}")
    print(f"  Goles en contra:     {stats['goles_contra']}")
    print(f"  Diferencia de goles: {stats['diferencia_goles']:+}")
    print(f"  % de victorias:      {stats['pct_victorias']}%")
    print(f"  Promedio goles:      {stats['prom_goles']}")

  📊 GUATEMALA
  Partidos jugados:    155
  Victorias:           62
  Empates:             36
  Derrotas:            57
  Goles a favor:       256
  Goles en contra:     201
  Diferencia de goles: +55
  % de victorias:      40.0%
  Promedio goles:      1.65


In [10]:
# Celda 8: Ver estadísticas de cualquier selección
# Cambia el nombre para ver otras selecciones

SELECCION = "Brazil"

stats = calcular_estadisticas(df_oficial, SELECCION)

if stats:
    print(f"{'='*40}")
    print(f"  📊 {stats['seleccion'].upper()}")
    print(f"{'='*40}")
    print(f"  Partidos jugados:    {stats['partidos']}")
    print(f"  Victorias:           {stats['victorias']}")
    print(f"  Empates:             {stats['empates']}")
    print(f"  Derrotas:            {stats['derrotas']}")
    print(f"  Goles a favor:       {stats['goles_favor']}")
    print(f"  Goles en contra:     {stats['goles_contra']}")
    print(f"  Diferencia de goles: {stats['diferencia_goles']:+}")
    print(f"  % de victorias:      {stats['pct_victorias']}%")
    print(f"  Promedio goles:      {stats['prom_goles']}")

  📊 BRAZIL
  Partidos jugados:    469
  Victorias:           283
  Empates:             100
  Derrotas:            86
  Goles a favor:       998
  Goles en contra:     416
  Diferencia de goles: +582
  % de victorias:      60.3%
  Promedio goles:      2.13


In [11]:
# Celda 8: Ver estadísticas de cualquier selección
# Cambia el nombre para ver otras selecciones

SELECCION = "France"

stats = calcular_estadisticas(df_oficial, SELECCION)

if stats:
    print(f"{'='*40}")
    print(f"  📊 {stats['seleccion'].upper()}")
    print(f"{'='*40}")
    print(f"  Partidos jugados:    {stats['partidos']}")
    print(f"  Victorias:           {stats['victorias']}")
    print(f"  Empates:             {stats['empates']}")
    print(f"  Derrotas:            {stats['derrotas']}")
    print(f"  Goles a favor:       {stats['goles_favor']}")
    print(f"  Goles en contra:     {stats['goles_contra']}")
    print(f"  Diferencia de goles: {stats['diferencia_goles']:+}")
    print(f"  % de victorias:      {stats['pct_victorias']}%")
    print(f"  Promedio goles:      {stats['prom_goles']}")

  📊 FRANCE
  Partidos jugados:    395
  Victorias:           227
  Empates:             89
  Derrotas:            79
  Goles a favor:       765
  Goles en contra:     359
  Diferencia de goles: +406
  % de victorias:      57.5%
  Promedio goles:      1.94


In [12]:
# Celda 9: Guardar la tabla de estadísticas
# Así no perdemos el trabajo y lo podemos usar en el siguiente notebook

import os

# Crear la carpeta "procesados" si no existe
RUTA_SALIDA = os.path.join("..", "datos", "procesados", "estadisticas_selecciones.csv")
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)

# Guardar
df_stats.to_csv(RUTA_SALIDA, index=True, index_label="ranking")

print(f"✅ Archivo guardado correctamente")
print(f"📁 Ubicación: datos/procesados/estadisticas_selecciones.csv")
print(f"📊 Selecciones guardadas: {len(df_stats)}")
print(f"\n🏆 Resumen final del notebook:")
print(f"   Partidos analizados:  {len(df_oficial):,}")
print(f"   Selecciones:          {len(df_stats)}")
print(f"   Torneos oficiales:    {df_oficial['tournament'].nunique()}")
print(f"   Años de historia:     {df_oficial['date'].dt.year.min()} → {df_oficial['date'].dt.year.max()}")

✅ Archivo guardado correctamente
📁 Ubicación: datos/procesados/estadisticas_selecciones.csv
📊 Selecciones guardadas: 223

🏆 Resumen final del notebook:
   Partidos analizados:  19,750
   Selecciones:          223
   Torneos oficiales:    12
   Años de historia:     1916 → 2026


In [ ]:
<